In [ ]:

import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)

In [ ]:
import pandas as pd

In [ ]:
leio_df = pd.read_csv('/run/media/victor/pessoal/mestrado/codigo/database_scripts/licence_filtered_leiomyoma_figure_caption.csv')

In [ ]:
leio_df

In [ ]:
leio_df["image_path"] = "/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/articles/" + leio_df["article_id"]  + "/" + leio_df["id"] + ".jpg" 

In [ ]:
leio_df = leio_df.drop_duplicates(subset=["image_path"])


In [ ]:

images_paths = leio_df["image_path"].values

In [ ]:
len(images_paths)

In [ ]:
import pickle

with open("/run/media/victor/pessoal/mestrado/codigo/model/V2_hist_occ.pkl", 'rb') as f:
    model_occ = pickle.load(f)

In [ ]:
import torchvision.models as models

model = models.convnext_tiny(weights='ConvNeXt_Tiny_Weights.IMAGENET1K_V1')
model.classifier=model.classifier[:-1]

In [ ]:
model.to("cuda").eval()

In [ ]:
device = "cuda"

In [ ]:
from torchvision import transforms


transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),        
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  
])


In [ ]:
from PIL import Image
import numpy as np

def is_majority_white(image_path, threshold=240, majority_percentage=0.5):
   
    try:

        img = Image.open(image_path).convert('L')
        

        img_array = np.array(img)
        

        total_pixels = img_array.size

        white_pixels = np.sum(img_array >= threshold)
        

        white_percentage = white_pixels / total_pixels
        
        return white_percentage > majority_percentage

    except FileNotFoundError:
        print(f"Error: The file '{image_path}' was not found.")
        return False
    except Exception as e:
        print(f"An error occurred: {e}")
        return False




In [ ]:
leio_df

In [ ]:
from tqdm import tqdm

whites = []
hist_paths = []
for img in tqdm(leio_df["image_path"].values):
    if is_majority_white(img, threshold=240, majority_percentage=0.5):
        whites.append(img)
    else:
        hist_paths.append(img)
  

In [ ]:
len(whites)

In [ ]:
leio_temp_df = leio_df[~ leio_df["image_path"].isin(whites)].copy().reset_index(drop=True)

In [ ]:
from dataloader_wrapper import get_features_article


if __name__ == "__main__":
    leio_temp_df = get_features_article(model, model_occ, transform, leio_temp_df)

In [ ]:
leio_df.isna().sum()

In [ ]:
leio_temp_df

In [ ]:
leio_df

In [ ]:
leio_temp_df["is_histopathology"].value_counts()

In [ ]:
leio_temp_df.to_csv("V2_leio_with_label.csv", index=False)

In [ ]:
import pandas as pd

leio_df = pd.read_csv("V2_leio_with_label.csv")

In [ ]:
leio_df['is_histopathology'].value_counts()

In [ ]:
from PIL import Image

Image.open(leio_df[~(leio_df["is_histopathology"])]['image_path'].values[25])

In [ ]:
leio_df

In [ ]:
leio_df[leio_df["is_histopathology"]]["image_path"].nunique()

In [ ]:
leio_df[leio_df["image_path"] == "/run/media/victor/pessoal/mestrado/codigo/datasets/leiomyoma_files/articles/PMC7046115/PAMJ-34-159-g001.jpg"]

In [ ]:
import shutil

for index, row in leio_df.iterrows():
    path = row['image_path']
    if row['is_histopathology']:
        shutil.copy(path, f"/run/media/victor/pessoal/mestrado/codigo/datasets/V2_leio_path/{row["article_id"]}_{row["id"]}.jpg")    